In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

: 

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# ==================================================
# Leitura dos dados processados
# ==================================================

input_path = Path("../data/processed/bank_marketing_processed.csv")
df = pd.read_csv(input_path)

print(f"Dataset carregado: {df.shape}")

# ==================================================
# Separação entre variáveis preditoras e alvo
# ==================================================

X = df.drop(columns=["y"])
y = df["y"]

# Caso a variável alvo ainda esteja como 'yes'/'no'
if y.dtype == "object":
    y = y.map({"no": 0, "yes": 1})
    
# ==================================================
# Divisão treino e teste
# ==================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Treino: {X_train.shape}")
print(f"Teste : {X_test.shape}")

: 

Criar ambientes de ofertas

In [ ]:
arms = {
    0: "Telefone",
    1: "Email",
    2: "SMS"
}


n_arms = len(arms)

print(arms)

In [ ]:
class LinearThompsonSampling:

    def __init__(
        self,
        n_features,
        n_arms,
        alpha=1
    ):

        self.n_features = n_features
        self.n_arms = n_arms
        self.alpha = alpha


        # Matrizes A
        self.A = [
            np.identity(n_features)
            for _ in range(n_arms)
        ]


        # Vetores b
        self.b = [
            np.zeros(n_features)
            for _ in range(n_arms)
        ]


    def choose_arm(self, x):

        samples = []


        for arm in range(self.n_arms):

            A_inv = np.linalg.inv(
                self.A[arm]
            )


            theta = A_inv @ self.b[arm]


            # amostra da distribuição posterior
            sampled_theta = np.random.multivariate_normal(
                theta,
                self.alpha * A_inv
            )


            reward = np.dot(
                sampled_theta,
                x
            )


            samples.append(reward)


        return np.argmax(samples)



    def update(
        self,
        arm,
        x,
        reward
    ):

        x = x.reshape(-1,1)


        self.A[arm] += (
            x @ x.T
        )


        self.b[arm] += (
            reward * x.flatten()
        )

In [ ]:
n_features = X_train.shape[1]


bandit = LinearThompsonSampling(
    n_features=n_features,
    n_arms=n_arms,
    alpha=1
)

In [ ]:
rewards = []

actions = []


for i in range(len(X_train)):


    x = X_train.iloc[i].values.astype(float)


    # escolhe canal
    arm = bandit.choose_arm(x)


    # recompensa observada
    reward = y_train.iloc[i]


    # atualiza aprendizado
    bandit.update(
        arm,
        x,
        reward
    )


    rewards.append(reward)
    actions.append(arm)



print(
    "Recompensa média:",
    np.mean(rewards)
)

In [ ]:
y_pred_bandit = []

chosen_channels = []


for i in range(len(X_test)):


    x = X_test.iloc[i].values.astype(float)


    arm = bandit.choose_arm(x)


    chosen_channels.append(
        arms[arm]
    )


    y_pred_bandit.append(
        arm
    )

In [ ]:
channel_conversion = {
    "Telefone": 0.30,
    "Email": 0.20,
    "SMS": 0.10
}


y_pred = []


for channel in chosen_channels:


    probability = channel_conversion[channel]


    prediction = (
        np.random.random()
        < probability
    )


    y_pred.append(
        int(prediction)
    )

In [ ]:
print(
    "Accuracy:",
    accuracy_score(
        y_test,
        y_pred
    )
)


print(
    "Precision:",
    precision_score(
        y_test,
        y_pred,
        zero_division=0
    )
)


print(
    "Recall:",
    recall_score(
        y_test,
        y_pred,
        zero_division=0
    )
)


print(
    "F1:",
    f1_score(
        y_test,
        y_pred,
        zero_division=0
    )
)

In [ ]:
pd.Series(
    chosen_channels
).value_counts()